# EXECUCAO MODELOS COMPARATIVOS - BATCH 1

Execute os modelos River (HAT, ARF, SRP), ACDWM e ERulesD2S nos chunks ja gerados pelo GBML Batch 1.

**Tempo estimado:** 4-6 horas (todos modelos em todos datasets)

## CELULA 1: Montar Drive e Setup Inicial

In [ ]:
from google.colab import drive
import os
import sys
from pathlib import Path

# Montar Drive
drive.mount('/content/drive')

# Definir paths
DRIVE_BASE = "/content/drive/Othercomputers/Laptop-CIn/Downloads/DSL-AG-hybrid"
WORK_DIR = "/content/DSL-AG-hybrid"

# Verificar se existe
if not os.path.exists(DRIVE_BASE):
    print(f"ERRO: Diretorio nao encontrado: {DRIVE_BASE}")
    raise FileNotFoundError(DRIVE_BASE)

print(f"Drive montado com sucesso!")
print(f"Diretorio base: {DRIVE_BASE}")

## CELULA 2: Copiar Repositorio para /content

In [ ]:
# Remover diretorio antigo se existir
if os.path.exists(WORK_DIR):
    print("Removendo diretorio antigo...")
    !rm -rf {WORK_DIR}

# Copiar do Drive (necessario para executar scripts Python)
print(f"Copiando repositorio...")
!cp -r {DRIVE_BASE} {WORK_DIR}

# Mudar para diretorio de trabalho
os.chdir(WORK_DIR)
print(f"Diretorio de trabalho: {os.getcwd()}")

# Verificar estrutura
print("\nArquivos principais:")
!ls -lh *.py | head -10

## CELULA 3: Instalar Dependencias

In [ ]:
# Instalar dependencias Python
print("Instalando dependencias...")
!pip install -q river scikit-learn pyyaml matplotlib seaborn pandas numpy

# Verificar instalacao
import river
from river import tree, ensemble, drift
print(f"\nRiver version: {river.__version__}")
print("Dependencias instaladas com sucesso!")

## CELULA 4: Clonar ACDWM Repository

In [ ]:
# Clonar repositorio ACDWM se nao existir
ACDWM_DIR = "/content/ACDWM"

if not os.path.exists(ACDWM_DIR):
    print("Clonando repositorio ACDWM...")
    !git clone https://github.com/jasonyanglu/ACDWM.git {ACDWM_DIR}
    print(f"ACDWM clonado em: {ACDWM_DIR}")
else:
    print(f"ACDWM ja existe em: {ACDWM_DIR}")

# Verificar arquivos
print("\nArquivos ACDWM:")
!ls -lh {ACDWM_DIR}/*.py | head -5

## CELULA 5: Verificar Chunks Disponiveis

In [ ]:
import pandas as pd

# Diretorio de resultados GBML
RESULTS_DIR = Path(WORK_DIR) / "experiments_6chunks_phase1_gbml" / "batch_1"

# Datasets do Batch 1
DATASETS = [
    "SEA_Abrupt_Simple",
    "AGRAWAL_Abrupt_Simple_Severe",
    "RBF_Abrupt_Severe",
    "HYPERPLANE_Abrupt_Simple",
    "STAGGER_Abrupt_Chain"
]

print("="*80)
print("VERIFICACAO DE CHUNKS DISPONIVEIS")
print("="*80)

for dataset in DATASETS:
    chunk_dir = RESULTS_DIR / dataset / "run_1" / "chunk_data"
    
    if chunk_dir.exists():
        train_chunks = list(chunk_dir.glob("chunk_*_train.csv"))
        test_chunks = list(chunk_dir.glob("chunk_*_test.csv"))
        
        print(f"\n{dataset}:")
        print(f"  Train chunks: {len(train_chunks)}")
        print(f"  Test chunks: {len(test_chunks)}")
        
        # Verificar tamanho do primeiro chunk de teste
        if test_chunks:
            first_test = sorted(test_chunks)[0]
            df = pd.read_csv(first_test)
            print(f"  Exemplo chunk shape: {df.shape}")
    else:
        print(f"\n{dataset}: CHUNKS NAO ENCONTRADOS!")
        print(f"  Path esperado: {chunk_dir}")

print("\n" + "="*80)

## CELULA 6: Executar River Models + ACDWM (1 Dataset)

Execute esta celula para processar UM dataset por vez (sem ERulesD2S):

In [ ]:
import subprocess
import time
from datetime import datetime

# ESCOLHER DATASET (mude conforme necessario)
DATASET_NAME = "SEA_Abrupt_Simple"  # Altere para outros datasets

# Configuracoes
BASE_DIR = Path(WORK_DIR) / "experiments_6chunks_phase1_gbml" / "batch_1"

print("="*80)
print(f"EXECUTANDO MODELOS COMPARATIVOS - {DATASET_NAME}")
print("="*80)
print(f"Inicio: {datetime.now()}")
print()

# Comando para executar run_comparative_on_existing_chunks.py
# Este script CARREGA os chunks salvos pelo GBML (nao regenera!)
cmd = [
    sys.executable,
    'run_comparative_on_existing_chunks.py',
    '--dataset', DATASET_NAME,
    '--base-dir', str(BASE_DIR),
    '--models', 'HAT', 'ARF', 'SRP',
    '--acdwm',
    '--acdwm-path', ACDWM_DIR,
    '--run', '1'
]

print("Comando:")
print(" ".join(cmd))
print()

start_time = time.time()

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        check=True
    )
    
    duration = time.time() - start_time
    
    print("="*80)
    print("EXECUCAO CONCLUIDA!")
    print("="*80)
    print(f"Duracao: {duration/60:.2f} minutos")
    print(f"\nStdout:\n{result.stdout}")
    
    # Listar resultados
    dataset_dir = BASE_DIR / DATASET_NAME / "run_1"
    print(f"\nDiretorio de resultados: {dataset_dir}")
    print("\nArquivos gerados:")
    !ls -lh {dataset_dir}/*.csv
    
except subprocess.CalledProcessError as e:
    print("="*80)
    print("ERRO NA EXECUCAO!")
    print("="*80)
    print(f"Codigo de erro: {e.returncode}")
    print(f"\nStderr:\n{e.stderr}")
    print(f"\nStdout:\n{e.stdout}")

## CELULA 7: Executar TODOS os Datasets (River + ACDWM + ERulesD2S)

**RECOMENDADO:** Execute esta celula para processar todos os 5 datasets sequencialmente com TODOS os modelos:

In [ ]:
import subprocess
import time
from datetime import datetime
import pandas as pd

DATASETS = [
    "SEA_Abrupt_Simple",
    "AGRAWAL_Abrupt_Simple_Severe",
    "RBF_Abrupt_Severe",
    "HYPERPLANE_Abrupt_Simple",
    "STAGGER_Abrupt_Chain"
]

BASE_DIR = Path(WORK_DIR) / "experiments_6chunks_phase1_gbml" / "batch_1"

print("="*80)
print("EXECUTANDO MODELOS COMPARATIVOS - BATCH 1 COMPLETO")
print("="*80)
print(f"Datasets: {len(DATASETS)}")
print(f"Modelos: HAT, ARF, SRP, ACDWM, ERulesD2S")
print(f"Inicio: {datetime.now()}")
print("="*80)

results_summary = []
total_start = time.time()

for idx, dataset in enumerate(DATASETS, 1):
    print(f"\n{'='*80}")
    print(f"[{idx}/{len(DATASETS)}] Processando: {dataset}")
    print('='*80)
    
    dataset_start = time.time()
    
    cmd = [
        sys.executable,
        'run_comparative_on_existing_chunks.py',
        '--dataset', dataset,
        '--base-dir', str(BASE_DIR),
        '--models', 'HAT', 'ARF', 'SRP',
        '--acdwm',
        '--acdwm-path', ACDWM_DIR,
        '--erulesd2s',
        '--erulesd2s-pop', '25',
        '--erulesd2s-gen', '50',
        '--run', '1'
    ]
    
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            check=True,
            timeout=7200  # 2 horas timeout por dataset (ERulesD2S pode demorar)
        )
        
        duration = time.time() - dataset_start
        status = "SUCESSO"
        
        print(f"\n{dataset}: {status} ({duration/60:.2f} min)")
        
        results_summary.append({
            'dataset': dataset,
            'status': status,
            'duration_min': duration/60
        })
        
    except subprocess.CalledProcessError as e:
        duration = time.time() - dataset_start
        status = "FALHA"
        
        print(f"\n{dataset}: {status} ({duration/60:.2f} min)")
        print(f"Erro: {e.stderr[:200]}")
        
        results_summary.append({
            'dataset': dataset,
            'status': status,
            'duration_min': duration/60,
            'error': str(e)[:100]
        })
        
    except subprocess.TimeoutExpired:
        print(f"\n{dataset}: TIMEOUT (>2h)")
        results_summary.append({
            'dataset': dataset,
            'status': 'TIMEOUT',
            'duration_min': 120
        })

total_duration = time.time() - total_start

print("\n" + "="*80)
print("RESUMO FINAL")
print("="*80)
print(f"Duracao total: {total_duration/3600:.2f} horas")
print(f"Datasets processados: {len([r for r in results_summary if r['status'] == 'SUCESSO'])}/{len(DATASETS)}")
print()

# Tabela resumo
summary_df = pd.DataFrame(results_summary)
print(summary_df.to_string(index=False))
print("="*80)

## CELULA 8: Instalar Java e Maven para ERulesD2S

In [ ]:
# Instalar Java 11 e Maven
print("Instalando Java 11 e Maven...")

!apt-get update -qq
!apt-get install -y -qq openjdk-11-jdk maven > /dev/null 2>&1

# Configurar JAVA_HOME
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'

# Verificar instalacao
!java -version
!mvn -version

print("\nJava e Maven instalados com sucesso!")

## CELULA 9: Setup ERulesD2S

In [ ]:
# Executar setup do ERulesD2S
print("Configurando ERulesD2S...")

# Verificar se setup_erulesd2s.py existe
setup_script = Path(WORK_DIR) / "setup_erulesd2s.py"

if setup_script.exists():
    !python setup_erulesd2s.py
    
    # Verificar se JAR foi criado
    if Path("erulesd2s.jar").exists():
        print("\nERulesD2S configurado com sucesso!")
        print("JAR encontrado: erulesd2s.jar")
        
        # Verificar JCLEC4 (obrigatorio)
        jclec_jar = Path("lib/JCLEC4-base-1.0-jar-with-dependencies.jar")
        if jclec_jar.exists():
            print(f"JCLEC4 encontrado: {jclec_jar}")
        else:
            print("AVISO: JCLEC4 nao encontrado!")
    else:
        print("ERRO: erulesd2s.jar nao foi criado!")
else:
    print(f"ERRO: {setup_script} nao encontrado!")
    print("Pulando setup ERulesD2S")

## CELULA 10: Executar Apenas ERulesD2S (Opcional)

Se voce quiser executar APENAS ERulesD2S separadamente (sem River/ACDWM):

In [ ]:
import subprocess
import time
from datetime import datetime

# ESCOLHER DATASET
DATASET_NAME = "SEA_Abrupt_Simple"  # Altere conforme necessario

BASE_DIR = Path(WORK_DIR) / "experiments_6chunks_phase1_gbml" / "batch_1"

print("="*80)
print(f"EXECUTANDO APENAS ERULESD2S - {DATASET_NAME}")
print("="*80)
print(f"Inicio: {datetime.now()}")
print()

start_time = time.time()

# Executar apenas ERulesD2S (sem River, sem ACDWM)
cmd = [
    sys.executable,
    'run_comparative_on_existing_chunks.py',
    '--dataset', DATASET_NAME,
    '--base-dir', str(BASE_DIR),
    '--models',  # Sem modelos River
    '--erulesd2s',
    '--erulesd2s-pop', '25',
    '--erulesd2s-gen', '50',
    '--run', '1'
]

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        check=True,
        timeout=3600
    )
    
    duration = time.time() - start_time
    
    print("="*80)
    print("EXECUCAO CONCLUIDA!")
    print("="*80)
    print(f"Duracao: {duration/60:.2f} minutos")
    print(f"\nStdout:\n{result.stdout}")
    
    # Verificar resultados
    dataset_dir = BASE_DIR / DATASET_NAME / "run_1"
    erulesd2s_file = dataset_dir / "erulesd2s_results.csv"
    
    if erulesd2s_file.exists():
        import pandas as pd
        df = pd.read_csv(erulesd2s_file)
        print(f"\nResultados ERulesD2S:")
        print(df)
        
except subprocess.CalledProcessError as e:
    print("="*80)
    print("ERRO NA EXECUCAO!")
    print("="*80)
    print(f"Codigo de erro: {e.returncode}")
    print(f"\nStderr:\n{e.stderr}")

## CELULA 11: Consolidar Resultados

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from scipy.stats import friedmanchisquare, wilcoxon
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path(WORK_DIR) / "experiments_6chunks_phase1_gbml" / "batch_1"

DATASETS = [
    "SEA_Abrupt_Simple",
    "AGRAWAL_Abrupt_Simple_Severe",
    "RBF_Abrupt_Severe",
    "HYPERPLANE_Abrupt_Simple",
    "STAGGER_Abrupt_Chain"
]

print("="*80)
print("CONSOLIDACAO DE RESULTADOS - BATCH 1 (COM GBML)")
print("="*80)

all_results = []

# ============================================================================
# CARREGAR RESULTADOS COMPARATIVOS (River, ACDWM, ERulesD2S)
# ============================================================================

for dataset in DATASETS:
    dataset_dir = BASE_DIR / dataset / "run_1"
    
    print(f"\n{dataset}:")
    
    # Listar todos os CSVs de resultados (exceto chunks)
    result_files = [
        f for f in dataset_dir.glob("*.csv")
        if "chunk_" not in f.name and "comparative" not in f.name
    ]
    
    for csv_file in result_files:
        try:
            df = pd.read_csv(csv_file)
            
            # CORRECAO PARA ERULESD2S: Converter formato
            if 'ERulesD2S' in csv_file.name or (df['model'] == 'ERulesD2S').any():
                print(f"  - {csv_file.name}: {len(df)} linhas (ERulesD2S - convertendo formato)")
                
                # ERulesD2S so tem test metrics (sem train)
                # Renomear colunas para padronizar
                if 'accuracy' in df.columns and 'test_accuracy' not in df.columns:
                    df.rename(columns={
                        'accuracy': 'test_accuracy',
                        'gmean': 'test_gmean',
                        'f1_weighted': 'test_f1'
                    }, inplace=True)
                    
                    # Adicionar colunas train vazias (ERulesD2S nao tem)
                    df['train_gmean'] = np.nan
                    df['train_accuracy'] = np.nan
                    df['train_f1'] = np.nan
                    
                    # Ajustar numeracao de chunks (ERulesD2S usa 1-6, outros usam 0-4)
                    if 'chunk' in df.columns:
                        df['chunk'] = df['chunk'] - 1  # 1-6 -> 0-5
            else:
                print(f"  - {csv_file.name}: {len(df)} linhas")
            
            # Adicionar coluna de dataset
            if 'dataset' not in df.columns:
                df['dataset'] = dataset
            
            all_results.append(df)
            
        except Exception as e:
            print(f"  ! Erro ao ler {csv_file.name}: {e}")

# ============================================================================
# CARREGAR RESULTADOS GBML (chunk_metrics.json)
# ============================================================================

print(f"\n{'='*60}")
print("Carregando resultados GBML (chunk_metrics.json)...")
print('='*60)

for dataset in DATASETS:
    dataset_dir = BASE_DIR / dataset / "run_1"
    chunk_metrics_file = dataset_dir / "chunk_metrics.json"
    
    if chunk_metrics_file.exists():
        try:
            import json
            with open(chunk_metrics_file, 'r') as f:
                gbml_data = json.load(f)
            
            # Converter para DataFrame
            df_gbml = pd.DataFrame(gbml_data)
            
            # Renomear colunas para padronizar (GBML ja tem train_ e test_)
            # Adicionar colunas faltantes
            df_gbml['model'] = 'GBML'
            df_gbml['dataset'] = dataset
            df_gbml['train_size'] = 1000
            df_gbml['test_size'] = 1000
            
            # Renomear test_f1 para test_f1 se necessario
            if 'test_f1' in df_gbml.columns:
                df_gbml['test_f1_orig'] = df_gbml['test_f1']
            
            # Adicionar train_accuracy e test_accuracy se nao existir
            if 'train_accuracy' not in df_gbml.columns:
                df_gbml['train_accuracy'] = np.nan
            if 'test_accuracy' not in df_gbml.columns:
                df_gbml['test_accuracy'] = np.nan
            if 'train_f1' not in df_gbml.columns:
                df_gbml['train_f1'] = np.nan
            if 'test_f1' not in df_gbml.columns:
                df_gbml['test_f1'] = df_gbml.get('test_f1_orig', np.nan)
            
            all_results.append(df_gbml)
            print(f"  - {dataset}: {len(df_gbml)} linhas (GBML)")
            
        except Exception as e:
            print(f"  ! Erro ao carregar GBML de {dataset}: {e}")
    else:
        print(f"  ! {dataset}: chunk_metrics.json NAO ENCONTRADO")

# ============================================================================
# CONSOLIDAR E SALVAR
# ============================================================================

if all_results:
    consolidated = pd.concat(all_results, ignore_index=True)
    
    # Salvar consolidado
    consolidated_file = BASE_DIR / "batch_1_all_models_with_gbml.csv"
    consolidated.to_csv(consolidated_file, index=False)
    
    print(f"\n{'='*80}")
    print("RESULTADOS CONSOLIDADOS (COM GBML)")
    print(f"{'='*80}")
    print(f"Total de linhas: {len(consolidated)}")
    print(f"Modelos: {sorted(consolidated['model'].unique())}")
    print(f"Datasets: {sorted(consolidated['dataset'].unique())}")
    print(f"\nArquivo salvo: {consolidated_file}")
    
    # ========================================================================
    # ESTATISTICAS RESUMIDAS
    # ========================================================================
    
    print(f"\n{'='*80}")
    print("ESTATISTICAS POR MODELO (Media +/- Desvio Padrao)")
    print(f"{'='*80}")
    
    summary = consolidated.groupby('model').agg({
        'train_gmean': ['mean', 'std', 'count'],
        'test_gmean': ['mean', 'std'],
        'train_accuracy': ['mean', 'std'],
        'test_accuracy': ['mean', 'std']
    }).round(4)
    
    print(summary)
    
    # ========================================================================
    # RANKING POR TEST G-MEAN
    # ========================================================================
    
    print(f"\n{'='*80}")
    print("RANKING POR TEST G-MEAN (Melhor para Pior)")
    print(f"{'='*80}")
    
    ranking = consolidated.groupby('model')['test_gmean'].mean().sort_values(ascending=False)
    for rank, (model, gmean) in enumerate(ranking.items(), 1):
        print(f"{rank}. {model:15s} test_gmean = {gmean:.4f}")
    
    # ========================================================================
    # COMPARACAO POR DATASET
    # ========================================================================
    
    print(f"\n{'='*80}")
    print("COMPARACAO POR DATASET (Test G-mean)")
    print(f"{'='*80}")
    
    pivot_table = consolidated.pivot_table(
        values='test_gmean',
        index='dataset',
        columns='model',
        aggfunc='mean'
    ).round(4)
    
    print(pivot_table)
    
    # ========================================================================
    # TESTES ESTATISTICOS
    # ========================================================================
    
    print(f"\n{'='*80}")
    print("TESTES ESTATISTICOS")
    print(f"{'='*80}")
    
    # Preparar dados para testes (matriz: datasets x modelos)
    models_to_compare = ['GBML', 'ACDWM', 'ARF', 'SRP', 'HAT']
    
    # Filtrar apenas modelos que temos dados
    models_to_compare = [m for m in models_to_compare if m in consolidated['model'].unique()]
    
    # Criar matriz de test_gmean por dataset e modelo
    data_matrix = []
    for model in models_to_compare:
        model_data = consolidated[consolidated['model'] == model].groupby('dataset')['test_gmean'].mean()
        data_matrix.append(model_data.values)
    
    data_matrix = np.array(data_matrix).T  # Transpor: datasets x modelos
    
    print(f"\nMatriz de dados (test_gmean):")
    print(f"Datasets: {len(data_matrix)} | Modelos: {len(models_to_compare)}")
    print(f"Modelos comparados: {models_to_compare}")
    
    # FRIEDMAN TEST
    print(f"\n--- Friedman Test ---")
    print("H0: Todos os modelos tem desempenho equivalente")
    print("H1: Pelo menos um modelo difere dos demais")
    
    try:
        stat, p_value = friedmanchisquare(*data_matrix.T)
        print(f"\nFriedman statistic: {stat:.4f}")
        print(f"p-value: {p_value:.6f}")
        
        alpha = 0.05
        if p_value < alpha:
            print(f"Resultado: REJEITA H0 (p < {alpha})")
            print("Conclusao: Existe diferenca significativa entre os modelos")
        else:
            print(f"Resultado: NAO REJEITA H0 (p >= {alpha})")
            print("Conclusao: Nao ha evidencia de diferenca entre os modelos")
    except Exception as e:
        print(f"Erro ao executar Friedman test: {e}")
    
    # WILCOXON SIGNED-RANK TEST (Pairwise)
    print(f"\n--- Wilcoxon Signed-Rank Tests (Pairwise) ---")
    print("Comparacao: GBML vs cada modelo")
    print()
    
    if 'GBML' in models_to_compare:
        gbml_idx = models_to_compare.index('GBML')
        gbml_scores = data_matrix[:, gbml_idx]
        
        for idx, model in enumerate(models_to_compare):
            if model == 'GBML':
                continue
            
            model_scores = data_matrix[:, idx]
            
            try:
                stat, p_value = wilcoxon(gbml_scores, model_scores, alternative='two-sided')
                
                mean_diff = np.mean(gbml_scores - model_scores)
                
                print(f"GBML vs {model:15s}: p={p_value:.6f}, diff={mean_diff:+.4f}", end="")
                if p_value < 0.05:
                    print(" *SIGNIFICATIVO*")
                else:
                    print()
                    
            except Exception as e:
                print(f"GBML vs {model:15s}: ERRO - {e}")
    
    print(f"\n{'='*80}")
    
else:
    print("\nNenhum resultado encontrado!")
    print("Verifique se os modelos foram executados com sucesso")

## CELULA 12: Gerar Plots Comparativos

Visualizacoes graficas para comparacao entre modelos:

---

## RESUMO DE USO

### Execucao Completa (Recomendada):

1. **CELULAS 1-3**: Setup inicial (5-10 min)
2. **CELULA 4**: Clonar ACDWM (1 min)
3. **CELULA 5**: Verificar chunks (30 seg)
4. **CELULAS 8-9**: Setup ERulesD2S (5 min)
5. **CELULA 7**: Executar TODOS modelos em TODOS datasets (4-6 horas)
6. **CELULA 11**: Consolidar resultados + GBML + Testes Estatisticos (1 min)
7. **CELULA 12**: Gerar plots comparativos (1 min)

### Tempo Total Estimado:

- River + ACDWM + ERulesD2S (5 datasets): 4-6 horas
- Somente River + ACDWM (5 datasets): 2-3 horas
- Somente ERulesD2S (5 datasets): 2-3 horas

### Arquivos Gerados:

**Resultados**:
- `batch_1_all_models_with_gbml.csv` - Consolidado completo
- `river_HAT_results.csv`, `river_ARF_results.csv`, etc. - Por modelo/dataset
- `erulesd2s_results.csv` - ERulesD2S por dataset

**Plots**:
- `boxplot_test_gmean.png` - Comparacao geral
- `barplot_mean_std.png` - Medias e desvios
- `heatmap_dataset_model.png` - Performance por dataset
- `lineplot_chunks_evolution.png` - Evolucao temporal
- `violinplot_distribution.png` - Distribuicoes completas
- `scatter_train_vs_test.png` - Overfitting analysis

### Observacoes:

- Nao feche o navegador durante a execucao
- Use Colab Pro para garantir 24h de runtime
- Os chunks GBML ja existentes serao reutilizados
- Resultados e plots salvos automaticamente no Drive

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from pathlib import Path

# Configurar estilo
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")

# Carregar dados consolidados
BASE_DIR = Path(WORK_DIR) / "experiments_6chunks_phase1_gbml" / "batch_1"
consolidated_file = BASE_DIR / "batch_1_all_models_with_gbml.csv"

if not consolidated_file.exists():
    print("ERRO: Execute a CELULA 11 primeiro para gerar o arquivo consolidado!")
else:
    df = pd.read_csv(consolidated_file)
    
    # Diretorio para salvar plots
    plots_dir = BASE_DIR / "comparative_plots"
    plots_dir.mkdir(exist_ok=True)
    
    print("="*80)
    print("GERANDO PLOTS COMPARATIVOS")
    print("="*80)
    
    # ========================================================================
    # PLOT 1: BOXPLOT - Test G-mean por Modelo
    # ========================================================================
    
    print("\n1. Boxplot - Test G-mean por Modelo")
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Filtrar modelos principais
    models_order = ['GBML', 'ACDWM', 'ARF', 'SRP', 'HAT', 'ERulesD2S']
    models_order = [m for m in models_order if m in df['model'].unique()]
    
    df_plot = df[df['model'].isin(models_order)]
    
    sns.boxplot(
        data=df_plot,
        x='model',
        y='test_gmean',
        order=models_order,
        ax=ax,
        palette='Set2'
    )
    
    ax.set_title('Comparacao de Modelos - Test G-mean (Batch 1)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Modelo', fontsize=12)
    ax.set_ylabel('Test G-mean', fontsize=12)
    ax.set_ylim(0, 1.05)
    ax.grid(axis='y', alpha=0.3)
    
    # Adicionar linha de media geral
    overall_mean = df_plot['test_gmean'].mean()
    ax.axhline(overall_mean, color='red', linestyle='--', alpha=0.5, label=f'Media Geral ({overall_mean:.3f})')
    ax.legend()
    
    plt.tight_layout()
    plot_file = plots_dir / "boxplot_test_gmean.png"
    plt.savefig(plot_file, dpi=300, bbox_inches='tight')
    print(f"   Salvo: {plot_file}")
    plt.close()
    
    # ========================================================================
    # PLOT 2: BARPLOT - Media e Desvio Padrao
    # ========================================================================
    
    print("\n2. Barplot - Media +/- Desvio Padrao")
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Calcular estatisticas
    stats = df_plot.groupby('model')['test_gmean'].agg(['mean', 'std']).reindex(models_order)
    
    x = np.arange(len(stats))
    ax.bar(x, stats['mean'], yerr=stats['std'], capsize=5, alpha=0.7, color=sns.color_palette('Set2', len(stats)))
    
    ax.set_xticks(x)
    ax.set_xticklabels(stats.index, rotation=0)
    ax.set_title('Comparacao de Modelos - Media +/- Desvio Padrao (Test G-mean)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Modelo', fontsize=12)
    ax.set_ylabel('Test G-mean', fontsize=12)
    ax.set_ylim(0, 1.05)
    ax.grid(axis='y', alpha=0.3)
    
    # Adicionar valores em cima das barras
    for i, (mean, std) in enumerate(zip(stats['mean'], stats['std'])):
        ax.text(i, mean + std + 0.02, f'{mean:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plot_file = plots_dir / "barplot_mean_std.png"
    plt.savefig(plot_file, dpi=300, bbox_inches='tight')
    print(f"   Salvo: {plot_file}")
    plt.close()
    
    # ========================================================================
    # PLOT 3: HEATMAP - Performance por Dataset e Modelo
    # ========================================================================
    
    print("\n3. Heatmap - Performance por Dataset")
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Criar pivot table
    pivot = df_plot.pivot_table(
        values='test_gmean',
        index='dataset',
        columns='model',
        aggfunc='mean'
    )[models_order]  # Reordenar colunas
    
    sns.heatmap(
        pivot,
        annot=True,
        fmt='.3f',
        cmap='RdYlGn',
        vmin=0,
        vmax=1,
        cbar_kws={'label': 'Test G-mean'},
        ax=ax
    )
    
    ax.set_title('Performance por Dataset e Modelo (Test G-mean)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Modelo', fontsize=12)
    ax.set_ylabel('Dataset', fontsize=12)
    
    plt.tight_layout()
    plot_file = plots_dir / "heatmap_dataset_model.png"
    plt.savefig(plot_file, dpi=300, bbox_inches='tight')
    print(f"   Salvo: {plot_file}")
    plt.close()
    
    # ========================================================================
    # PLOT 4: LINE PLOT - Evolucao por Chunk
    # ========================================================================
    
    print("\n4. Line Plot - Evolucao ao longo dos Chunks")
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    datasets = df['dataset'].unique()
    
    for idx, dataset in enumerate(datasets):
        if idx >= len(axes):
            break
        
        ax = axes[idx]
        df_dataset = df[(df['dataset'] == dataset) & (df['model'].isin(models_order))]
        
        for model in models_order:
            df_model = df_dataset[df_dataset['model'] == model].sort_values('chunk')
            if not df_model.empty:
                ax.plot(df_model['chunk'], df_model['test_gmean'], marker='o', label=model, linewidth=2)
        
        ax.set_title(dataset, fontsize=11, fontweight='bold')
        ax.set_xlabel('Chunk', fontsize=10)
        ax.set_ylabel('Test G-mean', fontsize=10)
        ax.set_ylim(0, 1.05)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8, loc='best')
    
    # Remover subplots vazios
    for idx in range(len(datasets), len(axes)):
        fig.delaxes(axes[idx])
    
    plt.suptitle('Evolucao de Performance ao Longo dos Chunks', fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plot_file = plots_dir / "lineplot_chunks_evolution.png"
    plt.savefig(plot_file, dpi=300, bbox_inches='tight')
    print(f"   Salvo: {plot_file}")
    plt.close()
    
    # ========================================================================
    # PLOT 5: VIOLIN PLOT - Distribuicao Completa
    # ========================================================================
    
    print("\n5. Violin Plot - Distribuicao de Performance")
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    sns.violinplot(
        data=df_plot,
        x='model',
        y='test_gmean',
        order=models_order,
        ax=ax,
        palette='Set2',
        inner='box'
    )
    
    ax.set_title('Distribuicao de Test G-mean por Modelo', fontsize=14, fontweight='bold')
    ax.set_xlabel('Modelo', fontsize=12)
    ax.set_ylabel('Test G-mean', fontsize=12)
    ax.set_ylim(0, 1.05)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plot_file = plots_dir / "violinplot_distribution.png"
    plt.savefig(plot_file, dpi=300, bbox_inches='tight')
    print(f"   Salvo: {plot_file}")
    plt.close()
    
    # ========================================================================
    # PLOT 6: COMPARACAO TRAIN vs TEST (apenas modelos com train metrics)
    # ========================================================================
    
    print("\n6. Scatter Plot - Train vs Test G-mean")
    
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # Filtrar modelos que tem train_gmean
    df_train_test = df_plot[df_plot['train_gmean'].notna()]
    
    for model in df_train_test['model'].unique():
        df_model = df_train_test[df_train_test['model'] == model]
        ax.scatter(
            df_model['train_gmean'],
            df_model['test_gmean'],
            label=model,
            s=100,
            alpha=0.6
        )
    
    # Linha diagonal (train = test)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Train = Test')
    
    ax.set_title('Train G-mean vs Test G-mean', fontsize=14, fontweight='bold')
    ax.set_xlabel('Train G-mean', fontsize=12)
    ax.set_ylabel('Test G-mean', fontsize=12)
    ax.set_xlim(0, 1.05)
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=10)
    ax.set_aspect('equal')
    
    plt.tight_layout()
    plot_file = plots_dir / "scatter_train_vs_test.png"
    plt.savefig(plot_file, dpi=300, bbox_inches='tight')
    print(f"   Salvo: {plot_file}")
    plt.close()
    
    # ========================================================================
    # RESUMO
    # ========================================================================
    
    print("\n" + "="*80)
    print("PLOTS GERADOS COM SUCESSO!")
    print("="*80)
    print(f"Diretorio: {plots_dir}")
    print("\nArquivos criados:")
    for plot_file in sorted(plots_dir.glob("*.png")):
        print(f"  - {plot_file.name}")
    print("="*80)